# 第 1 天练习 —— 用 GPT 解释本地代码

## 练习目标（理念）

做一个小工具：把本地文件 `scraper.py` 读进来，交给 **OpenAI Chat Completions**，让模型为「不熟悉代码库的开发者」逐函数讲解。

- **输入**：`scraper.py` 全文（拼进 user prompt）
- **输出**：Markdown 格式的代码说明（`display(Markdown(...))`）
- **模型**：`gpt-4o-mini`

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 环境变量 / `.env` | `load_dotenv` + 校验 `OPENAI_API_KEY` |
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么讲代码」，user 附上源码 |
| 笔记本展示 | `display(Markdown(...))` |

## 怎么跑

1. 同目录准备好 `scraper.py`，以及含 `OPENAI_API_KEY` 的 `.env`
2. 从上到下依次运行每个单元格（Shift+Enter）
3. 想换文件：改 `scraper_py` 路径，再重跑读取与调用两格

我的 fork：https://github.com/JaymanR/llm_engineering


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [ ]:
# ========== 环境：加载 .env 并校验 OPENAI_API_KEY ==========

# override=True：若进程里已有同名环境变量，也用 .env 里的值覆盖
load_dotenv(override=True)
# 从环境变量取出 API Key（字符串）；没有配置时得到 None
api_key = os.getenv('OPENAI_API_KEY')

# 分层体检：缺 key / 前缀不对 / 首尾有空白 —— 打印英文提示（与课程排错笔记本一致，勿改译）
if not api_key:
    # 完全没读到 key
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    # 有值但不是常见的 sk-proj- 前缀，可能拿错了 key
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    # strip 后变短 → 说明首尾有空格/制表符，容易导致鉴权失败
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 通过上述粗检，可以继续往下建客户端
    print("API key found and looks good so far!")

# 创建 OpenAI 客户端；默认会再读环境变量里的 OPENAI_API_KEY
openai = OpenAI()


In [ ]:
# ========== 读文件：把 scraper.py 全文装进字符串 ==========

# 目标文件名（相对当前工作目录）；要换文件只改这一处
scraper_py = "scraper.py"
# 先给空串兜底：读失败时后面仍能拼 prompt（只是没有源码）
script_content = ""
try:
    # 以 UTF-8 打开文本文件，读入全部内容到 script_content
    with open(scraper_py, "r", encoding="utf-8") as file:
        script_content = file.read()
    # 调试时可取消下一行注释，在笔记本里预览源码
    #print(script_content)
except FileNotFoundError:
    # 文件不存在：打印英文错误文案（保持原样，便于对照排错）
    print("The file 'scraper.py' does not exist.")
except Exception as e:
    # 其它 IO/解码错误：打印异常详情
    print(f"An error occurred: {e}")


In [ ]:
# ========== 调用 GPT：system/user → Chat Completions → Markdown 展示 ==========

# ----- 步骤 1：编写提示词（发给模型的英文 prompt 不翻译）-----

# system：定角色与输出格式——给新人讲代码，用 Markdown，且不要外包一层代码块
system_prompt = """
You are a helpful coding assistant that can summarize 
and explain the contents of existing code to a developer who is new to the codebase.
Respond in Markdown. Do not wrap markdown in a code block.
"""
# user：先写任务说明，再拼接刚刚读入的源码全文
user_prompt = """
Please explain the following code. Make sure that you describe each function and how it works: 
""" + script_content

# ----- 步骤 2：构造 messages 列表（Chat Completions 约定的角色对话）-----

# 两条消息：system 管风格，user 管具体代码；行尾 # fill this in 是原作者的占位提示，保留
messages = [{"role":"system", "content":system_prompt},
{"role":"user", "content":user_prompt}] # fill this in

# ----- 步骤 3：调用 OpenAI（非流式：等整段生成完再返回）-----
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)

# ----- 步骤 4：取出助手回复正文，用 Markdown 渲染到笔记本 -----
display(Markdown(response.choices[0].message.content))
